# v2 – Thí nghiệm khoá cơ chế và go/no-go (hướng tới ICCV)

Notebook **tự vá code trong runtime**, dùng chung `runs/`, `data/` trên Drive (checkpoint cũ được tái sử dụng để so sánh).
Run mới có tiền tố `runs/v2_…`. Kết quả ghi vào `{STORE}/results/v2`.

| Tầng | Thí nghiệm | Câu hỏi quyết định | Lượt |
|---|---|---|---|
| 0 | **I_conflict** trên E8 (không huấn luyện) | Xung đột trắc quang có trọng số chồng lấn có sắp thứ tự được ba cấu hình nhiễu E8? Chọn hoán vị cho E-A | 0 |
| 0 | **W_float^Σ, PSNR căn chỉnh** trên checkpoint cũ | Kết luận có giữ khi xét hình dạng Gaussian? | 0 |
| 2 | **T2 go/no-go**: 3DGS chuẩn (gsplat, SH3, khởi tạo SfM, 10k vòng) trên các scan DTU có sẵn | Hiện tượng có còn trong pipeline chuẩn? | 9/scan |
| 1 | **E13** oracle không clip, oracle + mặt nạ bão hoà | Ngoại lệ oracle trên scan83 có do phép cắt bão hoà? | 24 |
| 1 | **M2 GC-ILA** (căn chỉnh log dạng đóng + gauge + mặt nạ bão hoà) | Giữ được mức giảm floater của M1 mà không mất màu tuyệt đối? | 18 |
| 1 | **E12** thời điểm bù (visual hull, lưu snapshot) | Bù ngoại hình **trước** mật độ hoá có giảm floater? Floater hình thành khi nào? | 66 |
| 1 | **E-A** hoán vị cách gán nhiễu (cùng tập tham số) | Cấu trúc gán nhiễu lên camera có quyết định mức floater? | 24 |

**Tiêu chí quyết định được đặt trước khi chạy** (in ra ở mục 7):
- *Go* (T2): ΔW_float mức strong > 0 ở ≥ 80% số scan và trung vị ≥ 0.03.
- *Cơ chế thời điểm* (E12): warm-up giảm ΔW của affine ≥ 50% so với affine đồng thời, **và** lịch trễ không có affine
  (đối chứng) giảm ít hơn đáng kể.
- *GC-ILA*: ΔW ≤ 30% của biến thể gốc và PSNR trực tiếp không thấp hơn biến thể gốc quá 1 dB.
- *E-A*: tương quan hạng giữa I_conflict và ΔW dương trong cả hai scan.

Thứ tự: 1 → 2 → 3 → 4 → 5 (chạy lại được, run đã xong tự bỏ qua) → 6 → 7.

## 1. Setup

In [ ]:
import os, subprocess, pathlib, shutil, sys
from google.colab import drive, userdata
drive.mount('/content/drive')
STORE = '/content/drive/MyDrive/faithfulrecon_store'
REPO_URL = 'github.com/Huyrua27/faithfulrecon.git'
BRANCH = 'feature/huy-dtu-multiscan'
REPO = '/content/faithfulrecon'
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', f'https://{token}@{REPO_URL}', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
for d in ('runs', 'data'):
    p = pathlib.Path(REPO, d)
    if p.is_symlink(): p.unlink()
    elif p.exists(): shutil.rmtree(p)
    p.symlink_to(f'{STORE}/{d}')
os.chdir(REPO); sys.path.insert(0, REPO); sys.path.insert(0, f'{REPO}/scripts')
RES = f'{STORE}/results/v2'; os.makedirs(RES, exist_ok=True)
RES_OLD = f'{STORE}/results/final'
!git log --oneline -1
!pip -q install -r requirements.txt scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc

## 2. Vá code trong runtime (chạy lại sau mỗi lần khởi động lại runtime)
Gồm các bản vá cũ (tập kiểm định cố định, affine, SH bậc 3, cắt kích thước, nhiễu toàn cục, oracle, căn chỉnh dạng đóng,
visual hull) và **PATCH 5**: hoán vị nhiễu, nhiễu không clip, oracle theo clip, GC-ILA, mặt nạ bão hoà, warm-up hình học,
snapshot, khởi tạo SfM, sửa gradient mật độ hoá của backend gsplat. Cell phải in đủ `PATCH 1 OK`, `E5 patch OK`,
`E6 patch OK`, `PATCH 4 OK`, `PATCH 5 OK`.

In [ ]:
%%writefile scripts/floater_metrics.py
"""Floater and geometry metrics of trained DTU base models (one row per run_dir x seed).

python scripts/floater_metrics.py --run_dirs runs/dtu_scan24_v8 runs/photo_strong_dtu_scan24_v8 \
    --seeds 0 1 2 --out results/photometric/floater_metrics.csv

Metrics (normalized units; d = distance from a Gaussian mean to the nearest reference point):
  off_003, off_01   fraction of Gaussians with d > 0.03 / d > 0.1
  front_01          fraction of Gaussians with d > 0.1 that lie IN FRONT of the reference surface
                    (median over training views of z_gauss - z_surface < -tol)
  wfloat_01         share of held-out blending weight carried by Gaussians with d > 0.1
                    (the floaters that actually render)
  acc, comp         mean distance Gaussian -> reference and reference -> Gaussian
                    (Gaussians with opacity > 0.05)
  psnr, dssim       held-out views, ORIGINAL (unperturbed) appearance
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from frecon.camera import farthest_point_indices  # noqa: E402
from frecon.data.dtu import load_dtu_views, reference_points, split_views  # noqa: E402
from frecon.gaussians import nearest_dist  # noqa: E402
from frecon.measure import heldout_errors, render_views  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402


def project(P, cam):
    Xc = P @ cam.R.T + cam.t
    z = Xc[:, 2]
    u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
    v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
    ok = (z > 1e-4) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
    return z, u.clamp(0, cam.W - 1), v.clamp(0, cam.H - 1), ok


def zbuffer(ref, cam):
    z, u, v, ok = project(ref, cam)
    zb = torch.full((cam.H * cam.W,), float("inf"), device=ref.device)
    zb = zb.scatter_reduce(0, v[ok] * cam.W + u[ok], z[ok], "amin")
    return -F.max_pool2d(-zb.view(1, 1, cam.H, cam.W), 3, 1, 1)[0, 0]   # close small holes


@torch.no_grad()
def depth_side(means, views, tol):
    diffs = []
    for cam, mask, zb in views:
        z, u, v, ok = project(means, cam)
        zs = zb[v, u]
        ok &= mask[v, u] & torch.isfinite(zs)
        d = torch.full_like(z, float("nan"))
        d[ok] = z[ok] - zs[ok]
        diffs.append(d)
    return torch.nanmedian(torch.stack(diffs, 1), 1).values   # nan where no valid view


def split_eval(cams, n_train: int, n_test: int, test_pool: int = 0):
    """Training views as in training (FPS, nested in N). With test_pool > 0 the held-out views are
    taken outside the first max(n_train, test_pool) FPS cameras, so runs with any N <= test_pool share
    the same held-out views."""
    if test_pool <= 0:
        return split_views(cams, n_train, n_test)
    centers = torch.stack([c.center for c in cams])
    tr = farthest_point_indices(centers, n_train)
    pool = farthest_point_indices(centers, max(n_train, test_pool))
    rest = [i for i in range(len(cams)) if i not in pool]
    step = max(len(rest) // max(n_test, 1), 1)
    return tr, rest[::step][:n_test]


def strict_reference(sc, cams, masks, scale_mat, frac: float, seed: int = 0) -> torch.Tensor:
    """Reference points that lie inside the object mask in at least `frac` of the views whose image
    they project into (the default crop keeps points inside the mask of >= 1 view, which retains
    table / background points). Cached next to the default cache."""
    cache = Path(sc["ref_cache"]).with_name(f"ref_points_strict{int(round(frac * 100))}.npy")
    if cache.exists():
        return torch.from_numpy(np.load(cache)).float()
    from frecon.data.blender import _read_ply_xyz
    pts_w = _read_ply_xyz(Path(sc["stl_ply"])).numpy().astype(np.float64)
    inv = np.linalg.inv(scale_mat)
    pts = (pts_w @ inv[:3, :3].T) + inv[:3, 3]
    pts = pts[np.linalg.norm(pts, axis=1) < 1.5]
    n_in, n_frame = np.zeros(len(pts), np.int32), np.zeros(len(pts), np.int32)
    for cam, m in zip(cams, masks):
        R, t = cam.R.numpy().astype(np.float64), cam.t.numpy().astype(np.float64)
        pc = pts @ R.T + t
        z = pc[:, 2]
        ok = z > 1e-6
        u = np.floor(cam.fx * pc[:, 0] / np.where(ok, z, 1) + cam.cx).astype(np.int64)
        v = np.floor(cam.fy * pc[:, 1] / np.where(ok, z, 1) + cam.cy).astype(np.int64)
        inb = ok & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
        hit = np.zeros(len(pts), dtype=bool)
        hit[inb] = m[v[inb], u[inb]]
        n_frame += inb
        n_in += hit
    keep = (n_frame > 0) & (n_in >= frac * n_frame)
    print(f"strict reference ({frac:.2f}): kept {keep.sum()}/{len(pts)} points", flush=True)
    pts = pts[keep]
    mx = sc.get("max_ref_points", 200_000)
    if len(pts) > mx:
        pts = pts[np.random.default_rng(seed).choice(len(pts), mx, replace=False)]
    pts = pts.astype(np.float32)
    np.save(cache, pts)
    return torch.from_numpy(pts)


def load_eval_scene(cfg, dev, ref_mode: str = "default", strict_frac: float = 0.8, test_pool: int = 0):
    sc = cfg["scene"]
    assert sc.get("type") == "dtu", "floater_metrics.py supports DTU scenes only"
    cams, masks, scale_mat = load_dtu_views(sc["root"], sc.get("downscale", 4))
    tr, te = split_eval(cams, sc["n_train_views"], sc["n_test_views"], test_pool)
    if ref_mode == "strict":
        ref = strict_reference(sc, cams, masks, scale_mat, strict_frac).to(dev)
    else:
        ref = reference_points(sc.get("stl_ply"), scale_mat, cams, masks, sc.get("min_mask_views", 1),
                               sc.get("max_ref_points", 200_000), sc.get("ref_cache")).to(dev)
    views = []
    for i in tr:
        cam = cams[i].to(dev)
        views.append((cam, torch.from_numpy(masks[i]).to(dev), zbuffer(ref, cam)))
    return views, [cams[i].to(dev) for i in te], ref


def tau_tag(tau: float) -> str:
    """0.03 -> '003', 0.1 -> '01', 0.15 -> '015' (keeps the historical column names)."""
    return f"{tau:g}".replace("0.", "0", 1)


@torch.no_grad()
def metrics(model, views, test_cams, ref, rasterize, bg, tol, taus=(0.03, 0.05, 0.1, 0.15)):
    """W_float(tau) = sum_{k,p} sum_{i: d_i > tau} w_ik(p) / sum_{k,p} sum_i w_ik(p) over all pixels of
    the held-out views (torch renderer: exact blending weights, contributions < 1/255 dropped)."""
    means = model.means.detach()
    opac = model.opacities.detach().reshape(-1)
    d = nearest_dist(means, ref)
    med = depth_side(means, views, tol)
    contrib = torch.zeros(model.n, device=means.device)
    for cam in test_cams:
        contrib += model.render(rasterize, cam, bg, return_contrib=True)["contrib"]
    solid = opac > 0.05
    err = heldout_errors(render_views(model, test_cams, rasterize, bg), test_cams)
    out = {"n_gaussians": model.n}
    for tau in taus:
        far = d > tau
        t = tau_tag(tau)
        out[f"off_{t}"] = float(far.float().mean())
        out[f"wfloat_{t}"] = float(contrib[far].sum() / contrib.sum().clamp_min(1e-12))
        out[f"n_off_{t}"] = int(far.sum())                      # count, not a fraction
        out[f"opsum_off_{t}"] = float(opac[far].sum())          # total opacity of far Gaussians
        out[f"contrib_off_{t}"] = float(contrib[far].sum())     # absolute held-out weight
    off01 = d > 0.1
    out.update({
        "front_01": float(((med < -tol) & off01).sum() / off01.sum().clamp_min(1)),
        "contrib_total": float(contrib.sum()),
        "acc": float(d[solid].mean()),
        "comp": float(nearest_dist(ref, means[solid]).mean()),
        "psnr": err["psnr"], "dssim": err["dssim"],
    })
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--tol", type=float, default=0.03)
    ap.add_argument("--taus", type=float, nargs="+", default=[0.03, 0.05, 0.1, 0.15],
                    help="distance thresholds (normalized units, scene ~ unit sphere) for the sweep")
    ap.add_argument("--out", default="results/photometric/floater_metrics.csv")
    ap.add_argument("--ref_mode", choices=["default", "strict"], default="default",
                    help="strict: keep reference points inside the mask in >= --strict_frac of the views "
                         "they project into")
    ap.add_argument("--strict_frac", type=float, default=0.8)
    ap.add_argument("--test_pool", type=int, default=0,
                    help="> 0: held-out views outside the first max(N, test_pool) FPS cameras (same views "
                         "for every N <= test_pool)")
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        rasterize = get_rasterizer(cfg.get("renderer", "torch"))
        bg = torch.ones(3, device=dev)
        key = (cfg["scene"]["root"], cfg["scene"]["n_train_views"])
        if key not in cache:   # evaluation scene ignores `photometric`: original appearance
            cache = {key: load_eval_scene(cfg, dev, args.ref_mode, args.strict_frac, args.test_pool)}
        views, test_cams, ref = cache[key]
        photo = cfg["scene"].get("photometric") or {}
        for seed in args.seeds:
            ckpt = rd / f"seed_{seed}" / "base.pt"
            if not ckpt.exists():
                print(f"skip {ckpt} (missing)")
                continue
            model = Trainer.load(ckpt, TrainConfig(**cfg["train"]), rasterize, bg, dev).model
            tc, ic = cfg.get("train", {}), cfg.get("init", {})
            row = {"run": rd.name, "scan": Path(key[0]).name, "seed": seed, "N": key[1],
                   "exposure_stops": photo.get("exposure_stops", 0.0), "wb": photo.get("wb", 0.0),
                   "gamma": photo.get("gamma", 0.0), "photo_seed": photo.get("seed", 0),
                   "photo_mode": photo.get("mode", "isp"), "appearance": tc.get("appearance", "none"),
                   "lr_appearance": tc.get("lr_appearance", 1e-3), "sh_degree": tc.get("sh_degree", 0),
                   "cap": bool(tc.get("cap_scale_after_densify", False)), "init": ic.get("type", "random"),
                   "ref_mode": args.ref_mode, "test_pool": args.test_pool}
            row.update(metrics(model, views, test_cams, ref, rasterize, bg, args.tol, args.taus))
            rows.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)
            del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(args.out, index=False)
    print(f"-> {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/sfm_tri.py
"""Sparse points for SfM-style initialization from the CLEAN training views with known poses.

Replaces COLMAP's point_triangulator (features + matching + triangulation with fixed poses):
SIFT inside the object masks -> ratio-test matching for every view pair -> epipolar check with the
known fundamental matrix -> DLT triangulation -> cheirality, reprojection and triangulation-angle checks
-> voxel de-duplication. Positions are identical for every photometric level of a (scan, N) pair, so
paired comparisons isolate the effect of the perturbation on 3DGS optimization ("clean-SfM" init).

python scripts/sfm_tri.py --root data/dtu/scan63 --n_train 8 --out data/dtu/scan63/sfm_v8.npy
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import cv2
import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


def _skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])


def fundamental(K1, R1, t1, K2, R2, t2):
    R = R2 @ R1.T
    t = t2 - R @ t1
    return np.linalg.inv(K2).T @ (_skew(t) @ R) @ np.linalg.inv(K1)


def sampson(F, x1, x2):
    h1 = np.hstack([x1, np.ones((len(x1), 1))])
    h2 = np.hstack([x2, np.ones((len(x2), 1))])
    Fx1, Ftx2 = h1 @ F.T, h2 @ F
    num = np.sum(h2 * Fx1, 1) ** 2
    return num / (Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 + Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2 + 1e-12)


def triangulate_views(images, masks, Ks, Rs, ts, ratio=0.75, max_epi=1.5, max_reproj=1.5,
                      min_angle_deg=2.0, voxel=0.004, n_features=8000):
    """images: (H, W, 3) float in [0, 1]; masks: (H, W) bool; K, R, t in the convention of frecon.camera
    (pixel centres at index + 0.5). Returns (M, 3) float32 points."""
    sift = cv2.SIFT_create(nfeatures=n_features)
    feats = []
    for img, m in zip(images, masks):
        g = cv2.cvtColor((np.clip(img, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        kp, des = sift.detectAndCompute(g, m.astype(np.uint8) * 255)
        xy = np.float64([k.pt for k in kp]).reshape(-1, 2) + 0.5     # OpenCV pixel centres -> ours
        feats.append((xy, des))
    P = [K @ np.hstack([R, t[:, None]]) for K, R, t in zip(Ks, Rs, ts)]
    C = [-R.T @ t for R, t in zip(Rs, ts)]
    bf = cv2.BFMatcher(cv2.NORM_L2)
    out, n = [], len(images)
    for i in range(n):
        for j in range(i + 1, n):
            (xi, di), (xj, dj) = feats[i], feats[j]
            if di is None or dj is None or len(di) < 2 or len(dj) < 2:
                continue
            good = [a for a, b in (p for p in bf.knnMatch(di, dj, k=2) if len(p) == 2) if a.distance < ratio * b.distance]
            if len(good) < 8:
                continue
            x1 = xi[[g.queryIdx for g in good]]
            x2 = xj[[g.trainIdx for g in good]]
            F = fundamental(Ks[i], Rs[i], ts[i], Ks[j], Rs[j], ts[j])
            ok = sampson(F, x1, x2) < max_epi ** 2
            x1, x2 = x1[ok], x2[ok]
            if len(x1) == 0:
                continue
            X = cv2.triangulatePoints(P[i], P[j], x1.T, x2.T)
            X = (X[:3] / X[3]).T
            keep = np.ones(len(X), bool)
            for Pk, xk, Rk, tk in ((P[i], x1, Rs[i], ts[i]), (P[j], x2, Rs[j], ts[j])):
                z = X @ Rk[2] + tk[2]
                pr = np.hstack([X, np.ones((len(X), 1))]) @ Pk.T
                err = np.linalg.norm(pr[:, :2] / pr[:, 2:3] - xk, axis=1)
                keep &= (z > 1e-6) & (err < max_reproj)
            r1, r2 = X - C[i], X - C[j]
            cosang = np.sum(r1 * r2, 1) / (np.linalg.norm(r1, axis=1) * np.linalg.norm(r2, axis=1) + 1e-12)
            keep &= cosang < np.cos(np.radians(min_angle_deg))
            out.append(X[keep])
    if not out:
        return np.zeros((0, 3), np.float32)
    X = np.concatenate(out)
    X = X[np.linalg.norm(X, axis=1) < 1.5]
    _, idx = np.unique(np.floor(X / voxel).astype(np.int64), axis=0, return_index=True)
    return X[np.sort(idx)].astype(np.float32)


def scan_points(root: str, n_train: int, downscale: int = 2):
    """Triangulate from the FPS training views of a DTU scan (same views as training, clean images)."""
    from frecon.camera import farthest_point_indices
    from frecon.data.dtu import load_dtu_views
    cams, masks, _ = load_dtu_views(root, downscale)
    tr = farthest_point_indices(torch.stack([c.center for c in cams]), n_train)
    Ks = [np.array([[cams[i].fx, 0, cams[i].cx], [0, cams[i].fy, cams[i].cy], [0, 0, 1]]) for i in tr]
    return triangulate_views([cams[i].image.numpy() for i in tr], [masks[i] for i in tr], Ks,
                             [cams[i].R.numpy().astype(np.float64) for i in tr],
                             [cams[i].t.numpy().astype(np.float64) for i in tr])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--n_train", type=int, default=8)
    ap.add_argument("--out", required=True)
    ap.add_argument("--downscale", type=int, default=2)
    a = ap.parse_args()
    pts = scan_points(a.root, a.n_train, a.downscale)
    np.save(a.out, pts)
    print(f"{a.root}: {len(pts)} points -> {a.out}")

In [ ]:
%%writefile scripts/fm2.py
"""Evaluation v2 (strict reference, 16 shared held-out views), one row per run_dir x seed (x snapshot).

Always renders with the torch rasterizer (exact blending weights), whatever renderer was used for training.
Adds to floater_metrics.metrics:
  wsig_01       W_float with a shape-aware weight: per Gaussian, fraction of 7 sigma points (centre and
                centre +- 1 sigma along the 3 principal axes) farther than 0.1 from the reference
  psnr_aligned  held-out PSNR after a per-view, per-channel affine colour fit (removes a global colour shift)
With --snaps: light metrics (W_float(0.1), off(0.1), #Gaussians, PSNR) for every snap_XXXXX.pt and base.pt.

python scripts/fm2.py --run_dirs runs/v2_* --seeds 0 1 2 --out results/v2/x.csv [--snaps]
"""
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))

from floater_metrics import load_eval_scene, metrics  # noqa: E402
from frecon.gaussians import GaussianModel, nearest_dist  # noqa: E402
from frecon.measure import heldout_errors, render_views  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.render.torch_rast import quat_to_rotmat  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402

NAME = re.compile(r"(photo|v2)_(none|mild|strongng|strongs1|strongs2|strong)(.*)_dtu_scan(\d+)_v(\d+)$")


def parse(name: str) -> dict:
    m = NAME.match(name)
    if not m:
        return {"prefix": "?", "level": "?", "tags": name}
    return {"prefix": m[1], "level": m[2], "tags": m[3]}


@torch.no_grad()
def contrib(model, cams, ras, bg):
    c = torch.zeros(model.n, device=model.means.device)
    for cam in cams:
        c += model.render(ras, cam, bg, return_contrib=True)["contrib"]
    return c


@torch.no_grad()
def sigma_far(model, ref, tau=0.1):
    mu, s = model.means.detach(), model.scales.detach()
    R = quat_to_rotmat(model.quats.detach())
    far = (nearest_dist(mu, ref) > tau).float()
    for m in range(3):
        ax = R[:, :, m] * s[:, m:m + 1]
        for sg in (-1.0, 1.0):
            far += (nearest_dist(mu + sg * ax, ref) > tau).float()
    return far / 7


@torch.no_grad()
def aligned_psnr(model, cams, ras, bg):
    out = []
    for cam in cams:
        x = model.render(ras, cam, bg)["image"].clamp(0, 1).reshape(-1, 3)
        y = cam.image.reshape(-1, 3)
        fit = torch.empty_like(x)
        for ch in range(3):
            A = torch.stack([x[:, ch], torch.ones_like(x[:, ch])], 1)
            sol = torch.linalg.lstsq(A.cpu().double(), y[:, ch:ch + 1].cpu().double()).solution
            fit[:, ch] = (A @ sol.to(A).float())[:, 0]
        mse = ((fit.clamp(0, 1) - y) ** 2).mean()
        out.append(float(-10 * torch.log10(mse.clamp_min(1e-12))))
    return float(np.mean(out))


def light(model, test_cams, ref, ras, bg):
    cb = contrib(model, test_cams, ras, bg)
    d = nearest_dist(model.means.detach(), ref)
    far = d > 0.1
    err = heldout_errors(render_views(model, test_cams, ras, bg), test_cams)
    return {"wfloat_01": float(cb[far].sum() / cb.sum().clamp_min(1e-12)), "off_01": float(far.float().mean()),
            "n_gaussians": model.n, "psnr": err["psnr"]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--out", required=True)
    ap.add_argument("--snaps", action="store_true")
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ras, bg = get_rasterizer("torch"), torch.ones(3, device=dev)
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        sc, tc, ic = cfg["scene"], cfg.get("train", {}), cfg.get("init", {})
        key = (sc["root"], sc["n_train_views"])
        if key not in cache:
            cache = {key: load_eval_scene(cfg, dev, "strict", 0.8, 16)}
        views, test_cams, ref = cache[key]
        photo = sc.get("photometric") or {}
        info = dict(run=rd.name, scan=Path(sc["root"]).name, N=sc["n_train_views"], **parse(rd.name),
                    exposure_stops=photo.get("exposure_stops", 0.0), photo_seed=photo.get("seed", 0),
                    perm=str(photo.get("perm", "")), clip=bool(photo.get("clip", True)),
                    photo_mode=photo.get("mode", "isp"), appearance=tc.get("appearance", "none"),
                    lr_appearance=tc.get("lr_appearance", 1e-3), sh_degree=tc.get("sh_degree", 0),
                    sat_mask=bool(tc.get("sat_mask", False)), warmup=int(tc.get("warmup_geom", 0)),
                    init=ic.get("type", "random"), renderer=cfg.get("renderer", "torch"),
                    iterations=tc.get("iterations", 7000))
        for seed in args.seeds:
            ck = rd / f"seed_{seed}" / "base.pt"
            if not ck.exists():
                continue
            if args.snaps:
                for sp in sorted((rd / f"seed_{seed}").glob("snap_*.pt")) + [ck]:
                    if sp == ck:
                        model, step = Trainer.load(ck, TrainConfig(**tc), ras, bg, dev).model, info["iterations"]
                    else:
                        d = torch.load(sp, map_location=dev, weights_only=False)
                        model, step = GaussianModel.from_state_dict(d["params"], dev), int(d["step"])
                    rows.append(dict(info, seed=seed, step=step, **light(model, test_cams, ref, ras, bg)))
                    del model
                print(rd.name, seed, "snapshots done", flush=True)
            else:
                model = Trainer.load(ck, TrainConfig(**tc), ras, bg, dev).model
                row = dict(info, seed=seed, step=info["iterations"])
                row.update(metrics(model, views, test_cams, ref, ras, bg, 0.03, (0.05, 0.1)))
                cb = contrib(model, test_cams, ras, bg)
                row["wsig_01"] = float((cb * sigma_far(model, ref)).sum() / cb.sum().clamp_min(1e-12))
                row["psnr_aligned"] = aligned_psnr(model, test_cams, ras, bg)
                rows.append(row)
                print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()
                       if k in ("run", "seed", "wfloat_01", "wsig_01", "psnr", "psnr_aligned")}, flush=True)
                del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(args.out, index=False)
    print(f"-> {args.out} ({len(rows)} rows)")


if __name__ == "__main__":
    main()

In [ ]:
# === PATCH 1: tập kiểm định cố định (FRECON_TEST_POOL) và bù affine theo ảnh (train.appearance) ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

patch('frecon/data/dtu.py',
      '    rest = [i for i in range(len(cams)) if i not in train_idx]',
      '    import os   # PATCH_FIXED_TEST\n'
      '    ref_idx = farthest_point_indices(centers, max(n_train, int(os.environ.get("FRECON_TEST_POOL", 8))))\n'
      '    rest = [i for i in range(len(cams)) if i not in ref_idx]',
      'PATCH_FIXED_TEST')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    appearance: str = "none"        # PATCH_AFFINE: affine | oracle | cf_lin | cf_log (loss only)\n'
      '    lr_appearance: float = 1e-3\n', 'PATCH_AFFINE')
patch(t, '        self._reset_stats()\n\n    # ---- statistics used by densification',
      '        self._reset_stats()\n        self.app, self.app_opt = {}, None   # PATCH_APP_INIT\n\n'
      '    # ---- statistics used by densification', 'PATCH_APP_INIT')
patch(t, '        loss = photometric_loss(out["image"], cam.image, cfg.lambda_dssim)\n'
         '        self.opt.zero_grad(set_to_none=True)\n',
      '        loss = photometric_loss(self._appearance(out["image"], cam), cam.image, cfg.lambda_dssim)\n'
      '        self.opt.zero_grad(set_to_none=True)\n'
      '        if self.app_opt is not None:   # PATCH_APP_STEP\n'
      '            self.app_opt.zero_grad(set_to_none=True)\n', 'PATCH_APP_STEP')
patch(t, '        self.opt.step()\n        with torch.no_grad():  # keep quaternions',
      '        self.opt.step()\n        if self.app_opt is not None:   # PATCH_APP_OPT\n'
      '            self.app_opt.step()\n        with torch.no_grad():  # keep quaternions', 'PATCH_APP_OPT')
patch(t, '    def fit(self, cams,',
      '    def _appearance(self, img, cam):   # PATCH_APP_FN\n'
      '        if getattr(self.cfg, "appearance", "none") != "affine":\n'
      '            return img\n'
      '        if cam.name not in self.app:\n'
      '            A = torch.nn.Parameter(torch.eye(3, device=img.device))\n'
      '            b = torch.nn.Parameter(torch.zeros(3, device=img.device))\n'
      '            self.app[cam.name] = (A, b)\n'
      '            if self.app_opt is None:\n'
      '                self.app_opt = torch.optim.Adam([A, b], lr=self.cfg.lr_appearance)\n'
      '            else:\n'
      '                self.app_opt.add_param_group({"params": [A, b]})\n'
      '        A, b = self.app[cam.name]\n'
      '        return img @ A.T + b\n\n'
      '    def fit(self, cams,', 'PATCH_APP_FN')
print('PATCH 1 OK')


# === E5 PATCH: SH bậc 1-3 (màu phụ thuộc hướng nhìn), bật bằng train.sh_degree trong config ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

g = 'frecon/gaussians.py'
patch(g, '    def __init__(self, means, log_scales, quats, opacity_logits, color_logits):\n',
      '    def __init__(self, means, log_scales, quats, opacity_logits, color_logits, sh_rest=None):  # PATCH_SH_INIT\n', 'PATCH_SH_INIT')
patch(g, '            "color_logits": torch.nn.Parameter(color_logits.contiguous()),\n        }\n',
      '            "color_logits": torch.nn.Parameter(color_logits.contiguous()),\n        }\n'
      '        if sh_rest is not None:   # PATCH_SH_PARAM\n'
      '            self.params["sh_rest"] = torch.nn.Parameter(sh_rest.contiguous())\n', 'PATCH_SH_PARAM')
patch(g, '    def render(self, rasterize, cam, bg, features=None, opacities=None, **kw):\n'
         '        return rasterize(self.means, self.quats, self.scales,\n'
         '                         self.opacities if opacities is None else opacities,\n'
         '                         self.colors if features is None else features, cam, bg, **kw)\n',
      '    def enable_sh(self, degree: int):   # PATCH_SH_RENDER\n'
      '        if degree > 0 and "sh_rest" not in self.params:\n'
      '            k = (degree + 1) ** 2 - 1\n'
      '            self.params["sh_rest"] = torch.nn.Parameter(torch.zeros(self.n, k, 3, device=self.means.device))\n'
      '        return self\n\n'
      '    def view_colors(self, cam):\n'
      '        if "sh_rest" not in self.params:\n'
      '            return self.colors\n'
      '        d = self.means - cam.center.to(self.means)\n'
      '        d = d / d.norm(dim=-1, keepdim=True).clamp_min(1e-9)\n'
      '        B = _sh_basis(d, self.params["sh_rest"].shape[1])          # (N, K)\n'
      '        return torch.sigmoid(self.params["color_logits"] + (B[..., None] * self.params["sh_rest"]).sum(1))\n\n'
      '    def render(self, rasterize, cam, bg, features=None, opacities=None, **kw):\n'
      '        return rasterize(self.means, self.quats, self.scales,\n'
      '                         self.opacities if opacities is None else opacities,\n'
      '                         self.view_colors(cam) if features is None else features, cam, bg, **kw)\n', 'PATCH_SH_RENDER')
patch(g, '        return cls(*(sd[k] for k in PARAM_NAMES))\n',
      '        return cls(*(sd[k] for k in PARAM_NAMES), sh_rest=sd.get("sh_rest"))   # PATCH_SH_IO\n', 'PATCH_SH_IO')
patch(g, '        return GaussianModel(*(self.params[k].detach()[keep].clone() for k in PARAM_NAMES))\n',
      '        sh = self.params.get("sh_rest")   # PATCH_SH_SUBSET\n'
      '        return GaussianModel(*(self.params[k].detach()[keep].clone() for k in PARAM_NAMES),\n'
      '                             sh_rest=None if sh is None else sh.detach()[keep].clone())\n', 'PATCH_SH_SUBSET')
s = Path(g).read_text(encoding='utf-8')
if 'PATCH_SH_BASIS' not in s:
    s += '''

# PATCH_SH_BASIS: real SH basis without the DC term (constants as in 3DGS sh_utils)
def _sh_basis(d, k):
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    C1 = 0.4886025119029199
    C2 = (1.0925484305920792, -1.0925484305920792, 0.31539156525252005, -1.0925484305920792, 0.5462742152960396)
    C3 = (-0.5900435899266435, 2.890611442640554, -0.4570457994644658, 0.3731763325901154,
          -0.4570457994644658, 1.445305721320277, -0.5900435899266435)
    xx, yy, zz = x * x, y * y, z * z
    b = [-C1 * y, C1 * z, -C1 * x]
    if k > 3:
        b += [C2[0] * x * y, C2[1] * y * z, C2[2] * (2 * zz - xx - yy), C2[3] * x * z, C2[4] * (xx - yy)]
    if k > 8:
        b += [C3[0] * y * (3 * xx - yy), C3[1] * x * y * z, C3[2] * y * (4 * zz - xx - yy),
              C3[3] * z * (2 * zz - 3 * xx - 3 * yy), C3[4] * x * (4 * zz - xx - yy),
              C3[5] * z * (xx - yy), C3[6] * x * (xx - 3 * yy)]
    return torch.stack(b[:k], -1)
'''
    Path(g).write_text(s, encoding='utf-8'); print(g, ': patched (PATCH_SH_BASIS)')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n    sh_degree: int = 0   # PATCH_SH_CFG (model built with enable_sh in phase0)\n', 'PATCH_SH_CFG')
patch(t, '            [{"params": [model.params[k]], "lr": lrs[k], "name": k} for k in PARAM_NAMES], lr=0.0, eps=1e-15)',
      '            [{"params": [model.params[k]], "lr": lrs.get(k, cfg.lr_colors / 20), "name": k}   # PATCH_SH_OPT\n'
      '             for k in model.params], lr=0.0, eps=1e-15)', 'PATCH_SH_OPT')
patch(t, '                self._append({k: m.params[k].detach()[clone] for k in PARAM_NAMES})',
      '                self._append({k: m.params[k].detach()[clone] for k in m.params})   # PATCH_SH_CLONE', 'PATCH_SH_CLONE')
patch(t, '                for k in ("quats", "opacity_logits", "color_logits"):',
      '                for k in [k for k in m.params if k not in ("means", "log_scales")]:   # PATCH_SH_SPLIT', 'PATCH_SH_SPLIT')

patch('scripts/phase0.py',
      '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n',
      '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n'
      '    model.enable_sh(int(cfg["train"].get("sh_degree", 0)))   # PATCH_SH_PHASE0\n', 'PATCH_SH_PHASE0')

# self-test (CPU): SH0 unchanged, SH3 trains, densifies, save/load round trip
import importlib, torch, tempfile, os
import frecon.gaussians as G, frecon.train as T
importlib.reload(G); importlib.reload(T)
from frecon.render import get_rasterizer
from frecon.camera import Camera
ras, bg = get_rasterizer('torch'), torch.ones(3)
cams = [Camera(R=torch.eye(3), t=torch.tensor([0.1 * i, 0., 3.]), fx=40., fy=40., cx=16., cy=16., W=32, H=32,
               image=torch.rand(32, 32, 3) * 0.5 + 0.25, name=f'c{i}') for i in range(3)]
for deg in (0, 3):
    m = G.GaussianModel.random_in_box(400, 1.0, torch.Generator().manual_seed(0), 'cpu').enable_sh(deg)
    tr = T.Trainer(m, T.TrainConfig(iterations=30, densify_from=5, densify_until=25, densify_every=5,
                                    log_every=0, sh_degree=deg), 1.0, ras, bg)
    tr.fit(cams, iterations=30, log=None)
    p = os.path.join(tempfile.gettempdir(), f'sh{deg}.pt'); tr.save(p)
    tr2 = T.Trainer.load(p, T.TrainConfig(sh_degree=deg), ras, bg, 'cpu')
    same = torch.allclose(tr.model.render(ras, cams[0], bg)['image'], tr2.model.render(ras, cams[0], bg)['image'])
    has = 'sh_rest' in tr2.model.params
    print(f'SH{deg}: n={tr.model.n}, sh_rest={has}, reload identical={same}')
    assert same and has == (deg > 0)
print('E5 patch OK')


# === E6 PATCH: giu gioi han kich thuoc Gaussian sau pha mat do hoa (train.cap_scale_after_densify) ===
from pathlib import Path
def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')
t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n    cap_scale_after_densify: bool = False   # PATCH_CAP_CFG\n', 'PATCH_CAP_CFG')
patch(t, '        return float(loss)\n',
      '        if (densify and getattr(cfg, "cap_scale_after_densify", False)   # PATCH_CAP_SCALE\n'
      '                and self.step >= cfg.densify_until and self.step % cfg.densify_every == 0):\n'
      '            with torch.no_grad():\n'
      '                big = self.model.scales.max(dim=1).values > 0.1 * self.extent\n'
      '                if big.any():\n'
      '                    self.prune(~big)\n'
      '        return float(loss)\n', 'PATCH_CAP_SCALE')
# tu kiem tra (CPU)
import importlib, torch
import frecon.train as T; importlib.reload(T)
from frecon.gaussians import GaussianModel
from frecon.render import get_rasterizer
from frecon.camera import Camera
ras0, bg0 = get_rasterizer('torch'), torch.ones(3)
cams = [Camera(R=torch.eye(3), t=torch.tensor([0.1 * i, 0., 3.]), fx=40., fy=40., cx=16., cy=16., W=32, H=32,
               image=torch.rand(32, 32, 3), name=f'c{i}') for i in range(3)]
m = GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
tr = T.Trainer(m, T.TrainConfig(iterations=40, densify_from=5, densify_until=20, densify_every=5, log_every=0,
                                cap_scale_after_densify=True), 5.0, ras0, bg0)
with torch.no_grad():
    tr.model.params['log_scales'][:5] = 0.0   # 5 Gaussian scale 1.0 > nguong 0.5
tr.step = 20
tr.fit(cams, iterations=10, log=None)
n_big = int((tr.model.scales.max(1).values > 0.5).sum())
assert n_big == 0 and tr.model.n > 0, n_big
print('E6 patch OK, max scale', round(float(tr.model.scales.max()), 4))


# === PATCH 4: nhiễu toàn cục (E1b), bù chính xác (oracle, E4b), căn chỉnh dạng đóng (M1), khởi tạo visual hull (E9) ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

# (a) photometric.py: mode isp | pixel | global; lưu tham số nhiễu của từng ảnh cho oracle
append('frecon/data/photometric.py', '''

# PATCH_MODES_V2 (runtime patch from colab_rq1_final.ipynb)
ORACLE = {}   # camera name -> (params, mask) of the perturbation applied to that training view


def perturb_views(cams, masks, cfg: dict):
    params = sample_params(len(cams), cfg.get("exposure_stops", 0.0), cfg.get("wb", 0.0),
                           cfg.get("gamma", 0.0), cfg.get("seed", 0))
    mode = cfg.get("mode", "isp")
    if mode == "global":   # one transform for every view: the draw with the largest exposure change
        k0 = max(range(len(params)), key=lambda k: abs(np.log2(params[k]["gain"])))
        params = [params[k0]] * len(params)
    out = []
    for k, (cam, m, p) in enumerate(zip(cams, masks, params)):
        m = torch.as_tensor(np.asarray(m), device=cam.image.device).bool()
        img = apply(cam.image, m, p)
        if mode == "pixel":   # same per-pixel magnitude, random sign -> zero-mean, no per-view component
            g = torch.Generator().manual_seed(int(cfg.get("seed", 0)) * 1000 + k)
            sign = (torch.randint(0, 2, img.shape[:2], generator=g) * 2 - 1).to(img)[..., None]
            img = torch.where(m[..., None], (cam.image + sign * (img - cam.image)).clamp(0, 1), cam.image)
        ORACLE[cam.name] = (p, m.cpu())
        out.append(replace(cam, image=img))
    return out, params
''', 'PATCH_MODES_V2')

# (b) dtu.py: giữ mặt nạ của ảnh huấn luyện cho khởi tạo visual hull
patch('frecon/data/dtu.py', '    train = [cams[i] for i in tr]\n',
      '    train = [cams[i] for i in tr]\n    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n',
      'PATCH_HULL_MASKS')
append('frecon/data/dtu.py', '\n\nLAST_TRAIN_MASKS = []   # PATCH_HULL_GLOBAL\n', 'PATCH_HULL_GLOBAL')

# (c) train.py: appearance = oracle | cf_lin | cf_log (chỉ áp trong hàm mất mát)
t = 'frecon/train.py'
patch(t, '        if getattr(self.cfg, "appearance", "none") != "affine":\n            return img\n',
      '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n'
      '        if mode == "oracle":\n'
      '            return _oracle_appearance(img, cam)\n'
      '        if mode in ("cf_lin", "cf_log"):\n'
      '            return _closed_form_appearance(img, cam.image, mode)\n'
      '        if mode != "affine":\n'
      '            return img\n', 'PATCH_APP_MODES')
append(t, '''

# PATCH_APP_FUNCS (runtime patch from colab_rq1_final.ipynb)
def _oracle_appearance(img, cam):
    """Exact forward model of the perturbation of this training view (known parameters)."""
    from frecon.data.photometric import ORACLE
    if cam.name not in ORACLE:
        return img
    p, m = ORACLE[cam.name]
    m = m.to(img.device)
    scale = p["gain"] * torch.tensor(p["wb"], dtype=img.dtype, device=img.device)
    out = (img * scale).clamp(1e-4, 1.0) ** p["gamma"]
    return torch.where(m[..., None], out, img)


def _closed_form_appearance(img, target, mode):
    """Per-view, per-channel affine alignment of the render to the target, solved in closed form at
    every step (least squares, no gradient through the coefficients). cf_log works on log intensity,
    where gain, white balance and gamma of the ISP model are exactly affine per channel."""
    sel = (target < 0.99).any(-1)          # object pixels (background is composited white)
    if int(sel.sum()) < 64:
        return img
    if mode == "cf_log":
        x, y = img.clamp_min(1e-3).log(), target.clamp_min(1e-3).log()
    else:
        x, y = img, target
    with torch.no_grad():
        xs, ys = x[sel], y[sel]
        xm, ym = xs.mean(0), ys.mean(0)
        xc = xs - xm
        a = ((xc * (ys - ym)).mean(0) / xc.pow(2).mean(0).clamp_min(1e-6)).clamp(0.25, 4.0)
        b = ym - a * xm
    out = a * x + b
    if mode == "cf_log":
        out = out.exp()
    return torch.where(sel[..., None], out, img)
''', 'PATCH_APP_FUNCS')

# (d) phase0.py: init.type = hull (điểm ngẫu nhiên trong visual hull của mặt nạ huấn luyện)
ph0 = 'scripts/phase0.py'
patch(ph0, '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n',
      '    model = (_hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n'
      '             GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev))\n',
      'PATCH_HULL')
patch(ph0, 'def train_base(cfg, seed, train_cams, rasterize, bg, dev, path: Path):\n',
      '''def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN
    """Random points inside the visual hull of the training masks: inside the mask of every training
    view that sees them, and inside the image of at least half of the views."""
    import frecon.data.dtu as D
    masks = [torch.as_tensor(m, device=dev) for m in D.LAST_TRAIN_MASKS]
    assert len(masks) == len(cams), "hull init needs the DTU loader (masks of the training views)"
    n, h = cfg["init"]["n_points"], cfg["init"]["half_extent"]
    kept, total = [], 0
    for _ in range(200):
        P = (torch.rand(200_000, 3, generator=gen, device=dev) * 2 - 1) * h
        ok = torch.ones(len(P), dtype=torch.bool, device=dev)
        n_frame = torch.zeros(len(P), device=dev)
        for cam, m in zip(cams, masks):
            Xc = P @ cam.R.to(dev).T + cam.t.to(dev)
            z = Xc[:, 2]
            u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
            v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
            inb = (z > 1e-6) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
            hit = torch.zeros_like(ok)
            hit[inb] = m[v[inb], u[inb]]
            ok &= hit | ~inb
            n_frame += inb
        sel = P[ok & (n_frame >= max(2, len(cams) // 2))]
        kept.append(sel)
        total += len(sel)
        if total >= n:
            break
    P = torch.cat(kept)[:n]
    log(f"hull init: {len(P)} points")
    return GaussianModel.from_points(P, torch.rand(len(P), 3, generator=gen, device=dev))


def train_base(cfg, seed, train_cams, rasterize, bg, dev, path: Path):
''', 'PATCH_HULL_FN')

# ---- tự kiểm tra (CPU, vài giây) ----
import importlib, sys, torch, numpy as np
import frecon.data.photometric as ph, frecon.data.dtu as dtu, frecon.train as T, frecon.gaussians as G
for mod in (ph, dtu, G, T):
    importlib.reload(mod)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i):   # camera on a circle looking at the origin
    a = 2 * np.pi * i / 4
    c = torch.tensor([3 * np.sin(a), 0., 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams, masks = [], []
for i in range(4):
    R, t = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.tensor([0.6, 0.4, 0.3])
    m = np.zeros((32, 32), bool); m[8:24, 8:24] = True
    cams.append(Camera(R=R, t=t, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'c{i}')); masks.append(m)
cfgp = dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0)
glob_, _ = ph.perturb_views(cams, masks, dict(cfgp, mode='global'))
assert all(torch.equal(glob_[0].image, c.image) for c in glob_), 'global mode must give identical transforms'
isp, _ = ph.perturb_views(cams, masks, cfgp)
orc = T._oracle_appearance(torch.stack([c.image for c in cams])[0], cams[0])
assert torch.allclose(orc, isp[0].image, atol=1e-3), 'oracle must reproduce the perturbed image'
for mode in ('cf_lin', 'cf_log'):
    al = T._closed_form_appearance(cams[0].image, isp[0].image, mode)
    assert (al - isp[0].image).abs().max() < 0.02, (mode, float((al - isp[0].image).abs().max()))
for app in ('oracle', 'cf_log', 'cf_lin', 'affine'):
    m0 = G.GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
    tr = T.Trainer(m0, T.TrainConfig(iterations=20, densify_from=5, densify_until=15, densify_every=5,
                                     log_every=0, appearance=app), 1.0, ras, bg)
    tr.fit(isp, iterations=20, log=None)
    print(f'{app}: 20 steps OK, n={tr.model.n}')
dtu.LAST_TRAIN_MASKS[:] = masks
hm = phase0._hull_init({'init': {'n_points': 500, 'half_extent': 1.0}}, cams, torch.Generator().manual_seed(0), 'cpu')
assert hm.n == 500 and float(hm.means.abs().max()) < 0.8, float(hm.means.abs().max())
print('PATCH 4 OK (global, oracle, closed-form, hull)')


# === PATCH 5 (v2): hoán vị nhiễu, nhiễu không clip, oracle theo clip, GC-ILA, mặt nạ bão hoà, warm-up, snapshot,
# ===               khởi tạo SfM, sửa gradient densification của backend gsplat
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

# (a) photometric.py: perturb_views V3 = V2 + hoán vị tham số giữa các ảnh (perm) + nhiễu không clip (clip: false)
append('frecon/data/photometric.py', '''

# PATCH_MODES_V3 (runtime patch from colab_v2_mechanism.ipynb); overrides the V2 definition above
def perturb_views(cams, masks, cfg: dict):
    params = sample_params(len(cams), cfg.get("exposure_stops", 0.0), cfg.get("wb", 0.0),
                           cfg.get("gamma", 0.0), cfg.get("seed", 0))
    perm = cfg.get("perm")
    if perm is not None:          # same set of transforms, different assignment to the cameras
        assert sorted(perm) == list(range(len(params))), perm
        params = [params[j] for j in perm]
    mode = cfg.get("mode", "isp")
    clip = bool(cfg.get("clip", True))
    if mode == "global":
        k0 = max(range(len(params)), key=lambda k: abs(np.log2(params[k]["gain"])))
        params = [params[k0]] * len(params)
    out = []
    for k, (cam, m, p) in enumerate(zip(cams, masks, params)):
        m = torch.as_tensor(np.asarray(m), device=cam.image.device).bool()
        scale = p["gain"] * torch.tensor(p["wb"], dtype=cam.image.dtype, device=cam.image.device)
        v = cam.image * scale
        v = v.clamp(0, 1) if clip else v.clamp_min(0)
        img = torch.where(m[..., None], v ** p["gamma"], cam.image)
        if mode == "pixel":
            g = torch.Generator().manual_seed(int(cfg.get("seed", 0)) * 1000 + k)
            sign = (torch.randint(0, 2, img.shape[:2], generator=g) * 2 - 1).to(img)[..., None]
            img = torch.where(m[..., None], (cam.image + sign * (img - cam.image)).clamp(0, 1), cam.image)
        ORACLE[cam.name] = (p, m.cpu(), clip)
        out.append(replace(cam, image=img))
    return out, params
''', 'PATCH_MODES_V3')

# (b) dtu.py: mặt nạ vật thể theo tên camera (để mặt nạ bão hoà không loại nền trắng)
patch('frecon/data/dtu.py', '    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n',
      '    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n'
      '    MASK_BY_NAME.update({cams[i].name: masks[i] for i in tr})   # PATCH_MASK_NAME\n', 'PATCH_MASK_NAME')
append('frecon/data/dtu.py', '\n\nMASK_BY_NAME = {}   # PATCH_MASK_NAME_GLOBAL\n', 'PATCH_MASK_NAME_GLOBAL')

# (c) train.py
t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    sat_mask: bool = False      # PATCH_V2_CFG: drop saturated object pixels of the target from the loss\n'
      '    sat_thr: float = 0.98\n'
      '    warmup_geom: int = 0        # first k iterations: geometry frozen, no densification\n'
      '    gc_eta: float = 0.2         # GC-ILA: EMA rate of the per-view coefficients\n'
      '    gc_gauge: bool = True       # GC-ILA: geometric mean of a = 1, mean of b = 0 over views\n'
      '    snapshots: tuple = ()       # iterations at which phase0 saves snap_XXXXX.pt\n', 'PATCH_V2_CFG')
patch(t, '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n',
      '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n'
      '        if mode == "gc_log":   # PATCH_GC_MODE\n'
      '            return _gc_appearance(self, img, cam)\n', 'PATCH_GC_MODE')
patch(t, '        loss = photometric_loss(self._appearance(out["image"], cam), cam.image, cfg.lambda_dssim)\n',
      '        loss = _loss_v2(self._appearance(out["image"], cam), cam.image, cfg, cam)   # PATCH_LOSS_V2\n', 'PATCH_LOSS_V2')
patch(t, '        self._set_means_lr(means_lr)\n',
      '        frozen = self.step < getattr(cfg, "warmup_geom", 0)   # PATCH_WARMUP\n'
      '        if frozen:\n'
      '            means_lr, densify = 0.0, False\n'
      '        for g in self.opt.param_groups:\n'
      '            if g["name"] in ("log_scales", "quats"):\n'
      '                g.setdefault("lr0", g["lr"])\n'
      '                g["lr"] = 0.0 if frozen else g["lr0"]\n'
      '        self._set_means_lr(means_lr)\n', 'PATCH_WARMUP')
patch(t, '            out["means2d"].retain_grad()\n',
      '            out.get("means2d_full", out["means2d"]).retain_grad()   # PATCH_GSPLAT_GRAD\n', 'PATCH_GSPLAT_GRAD')
patch(t, '                g = out["means2d"].grad\n',
      '                g = out["means2d"].grad   # PATCH_GSPLAT_GRAD2\n'
      '                if "means2d_full" in out and out["means2d_full"].grad is not None:\n'
      '                    g = out["means2d_full"].grad[0]\n', 'PATCH_GSPLAT_GRAD2')
append(t, '''

# PATCH_V2_FUNCS (runtime patch from colab_v2_mechanism.ipynb)
from .losses import ssim_map as _ssim_map
_MASK_CACHE = {}


def _object_mask(cam, device):
    from frecon.data import dtu as _dtu
    m = getattr(_dtu, "MASK_BY_NAME", {}).get(cam.name)
    if m is None:
        return None
    key = (cam.name, str(device))
    if key not in _MASK_CACHE:
        _MASK_CACHE[key] = torch.as_tensor(m, device=device).bool()
    return _MASK_CACHE[key]


def _loss_v2(pred, gt, cfg, cam):
    """Photometric loss; with sat_mask, object pixels whose target is saturated (any channel >= sat_thr) are
    excluded (the composited white background is kept)."""
    if not getattr(cfg, "sat_mask", False):
        return photometric_loss(pred, gt, cfg.lambda_dssim)
    m = _object_mask(cam, gt.device)
    sat = (gt >= getattr(cfg, "sat_thr", 0.98)).any(-1)
    w = (~(sat & m) if m is not None else torch.ones_like(sat)).float()
    den = w.sum().clamp_min(1.0)
    l1 = ((pred - gt).abs().mean(-1) * w).sum() / den
    ss = ((1 - _ssim_map(pred, gt)) * w).sum() / den
    return (1 - cfg.lambda_dssim) * l1 + cfg.lambda_dssim * ss


def _oracle_appearance(img, cam):   # V3: honours the clip flag of the perturbation (overrides V2)
    from frecon.data.photometric import ORACLE
    if cam.name not in ORACLE:
        return img
    e = ORACLE[cam.name]
    p, m, clip = e[0], e[1], (e[2] if len(e) > 2 else True)
    m = m.to(img.device)
    scale = p["gain"] * torch.tensor(p["wb"], dtype=img.dtype, device=img.device)
    v = (img * scale).clamp_min(1e-4)
    if clip:
        v = v.clamp(max=1.0)
    return torch.where(m[..., None], v ** p["gamma"], img)


def _gc_appearance(tr, img, target_cam):
    """GC-ILA: per-view, per-channel affine alignment in log intensity solved in closed form at every step
    (weighted LS, ridge towards slope 1, saturated / dark target pixels excluded), smoothed by an EMA, then
    projected on the gauge {geometric mean of a = 1, mean of b = 0 over the views}. No gradient through the
    coefficients."""
    cfg, target = tr.cfg, target_cam.image
    st = tr.__dict__.setdefault("gc", {})
    dev = img.device
    obj = (target < 0.99).any(-1)                       # object pixels (background composited white)
    x, y = img.clamp_min(1e-3).log(), target.clamp_min(1e-3).log()
    a_prev, b_prev = st.get(target_cam.name, (torch.ones(3, device=dev), torch.zeros(3, device=dev)))
    with torch.no_grad():
        w = (obj[..., None] & (target > 0.02) & (target < getattr(cfg, "sat_thr", 0.98))).float()
        n = w.sum((0, 1))
        if bool((n > 64).all()):
            xm, ym = (w * x).sum((0, 1)) / n, (w * y).sum((0, 1)) / n
            xc, yc = x - xm, y - ym
            cov, var = (w * xc * yc).sum((0, 1)) / n, (w * xc * xc).sum((0, 1)) / n
            lam = 1e-3
            a = ((cov + lam) / (var + lam)).clamp(0.5, 2.0)   # ridge towards a = 1
            b = ym - a * xm
            eta = getattr(cfg, "gc_eta", 0.2)
            st[target_cam.name] = ((1 - eta) * a_prev + eta * a, (1 - eta) * b_prev + eta * b)
        else:
            st[target_cam.name] = (a_prev, b_prev)
        if getattr(cfg, "gc_gauge", True) and len(st) > 1:
            A = torch.stack([v[0] for v in st.values()])
            B = torch.stack([v[1] for v in st.values()])
            ga, gb = A.log().mean(0).exp(), B.mean(0)
            for k in list(st):
                st[k] = (st[k][0] / ga, st[k][1] - gb)
        a, b = st[target_cam.name]
    return torch.where(obj[..., None], (a * x + b).exp(), img)
''', 'PATCH_V2_FUNCS')

# (d) gsplat backend: trả về tensor means2d gốc để retain_grad có gradient (view [0] không nằm trên đồ thị)
patch('frecon/render/gsplat_rast.py',
      '    out = {"image": colors[0], "alpha": alphas[0, ..., 0], "means2d": means2d,\n',
      '    out = {"means2d_full": meta["means2d"],   # PATCH_GSPLAT_FULL\n'
      '           "image": colors[0], "alpha": alphas[0, ..., 0], "means2d": means2d,\n', 'PATCH_GSPLAT_FULL')

# (e) phase0.py: snapshot theo vòng lặp, khởi tạo SfM (điểm triangulate sẵn, màu lấy từ ảnh huấn luyện)
ph0 = 'scripts/phase0.py'
patch(ph0, '    tr.fit(train_cams, seed=seed, log=lambda m: log(m))\n',
      '    snaps = {int(x) for x in (cfg["train"].get("snapshots") or [])}   # PATCH_SNAP\n'
      '    def _snap(it):\n'
      '        if it in snaps:\n'
      '            torch.save({"params": tr.model.state_dict(), "step": it, "extent": tr.extent},\n'
      '                       path.parent / f"snap_{it:05d}.pt")\n'
      '    tr.fit(train_cams, seed=seed, log=lambda m: log(m), callback=_snap if snaps else None)\n', 'PATCH_SNAP')
patch(ph0, '    model = (_hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n',
      '    model = (_sfm_init(cfg, train_cams, dev) if cfg["init"].get("type") == "sfm" else   # PATCH_SFM\n'
      '             _hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n',
      'PATCH_SFM')
patch(ph0, 'def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN\n',
      '''def _sfm_init(cfg, cams, dev):   # PATCH_SFM_FN
    """Points triangulated beforehand (scripts/sfm_tri.py, clean training views, known poses); colours = mean of
    the (possibly perturbed) training images over the views that see each point, as with COLMAP colours."""
    import numpy as np
    P = torch.from_numpy(np.load(cfg["init"]["points"])).float().to(dev)
    cols = torch.zeros_like(P)
    cnt = torch.zeros(len(P), 1, device=dev)
    for cam in cams:
        Xc = P @ cam.R.to(dev).T + cam.t.to(dev)
        z = Xc[:, 2]
        u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
        v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
        ok = (z > 1e-6) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
        cols[ok] += cam.image.to(dev)[v[ok], u[ok]]
        cnt[ok] += 1
    cols = torch.where(cnt > 0, cols / cnt.clamp_min(1), torch.full_like(cols, 0.5))
    log(f"sfm init: {len(P)} points")
    return GaussianModel.from_points(P, cols.clamp(0.02, 0.98))


def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN
''', 'PATCH_SFM_FN')

# (f) phase0.py: bỏ bước tính tín hiệu sau huấn luyện khi chỉ cần base model (v2 không dùng; với gsplat bước này lỗi
#     vì autograd.grad theo view means2d[0] không nằm trên đồ thị)
patch(ph0, '    sig = cached(sdir / "signals.pt", lambda: {k: v.cpu() for k, v in compute_signals(\n',
      '    if cfg.get("base_only") and cfg.get("skip_signals", False):   # PATCH_SKIP_SIGNALS\n'
      '        log(f"seed {seed}: base model ready (signals skipped)")\n'
      '        return None\n'
      '    sig = cached(sdir / "signals.pt", lambda: {k: v.cpu() for k, v in compute_signals(\n', 'PATCH_SKIP_SIGNALS')

# ---- tự kiểm tra (CPU, khoảng 1 phút) ----
import importlib, sys, tempfile, torch, numpy as np
import frecon.data.photometric as ph, frecon.data.dtu as dtu, frecon.train as T, frecon.gaussians as G
for mod in (ph, dtu, G, T):
    importlib.reload(mod)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i, n=4):
    a = 2 * np.pi * i / n
    c = torch.tensor([3 * np.sin(a), 0.3 * (i % 2), 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams, masks = [], []
gimg = torch.Generator().manual_seed(1)
for i in range(4):
    R, t = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.rand(16, 16, 3, generator=gimg) * 0.8 + 0.1
    m = np.zeros((32, 32), bool); m[8:24, 8:24] = True
    cams.append(Camera(R=R, t=t, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'v2c{i}')); masks.append(m)
dtu.MASK_BY_NAME.update({c.name: m for c, m in zip(cams, masks)})
base = dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0)
p_id, P0 = ph.perturb_views(cams, masks, base)
p_pm, P1 = ph.perturb_views(cams, masks, dict(base, perm=[2, 0, 3, 1]))
assert P1[0] == P0[2] and P1[3] == P0[1], 'perm'
nc, _ = ph.perturb_views(cams, masks, dict(base, clip=False, exposure_stops=2.0))
assert float(max(c.image.max() for c in nc)) > 1.0, 'noclip should exceed 1'
for c_clean, c_noisy in zip(cams, nc):
    assert torch.allclose(T._oracle_appearance(c_clean.image, c_noisy), c_noisy.image, atol=1e-4), 'oracle noclip'
pc, _ = ph.perturb_views(cams, masks, base)
assert torch.allclose(T._oracle_appearance(cams[0].image, pc[0]), pc[0].image, atol=1e-3), 'oracle clip'
# loss with saturation mask: background kept, saturated object pixels dropped
tc = T.TrainConfig(sat_mask=True)
pred = pc[0].image.clone(); pred[0, 0] = 0.0                                  # background error must count
assert float(T._loss_v2(pred, pc[0].image, tc, pc[0])) > 0
# GC-ILA, warm-up (geometry frozen), snapshots, SfM init through phase0.train_base
for app in ('gc_log', 'oracle', 'affine'):
    m0 = G.GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
    tr = T.Trainer(m0, T.TrainConfig(iterations=24, densify_from=5, densify_until=20, densify_every=5, log_every=0,
                                     appearance=app, sat_mask=True, warmup_geom=10), 1.0, ras, bg)
    mu0 = tr.model.means.detach().clone(); n0 = tr.model.n
    tr.fit(pc, iterations=10, log=None)
    assert torch.equal(tr.model.means.detach(), mu0) and tr.model.n == n0, f'{app}: warm-up must freeze geometry'
    tr.fit(pc, iterations=14, log=None)
    assert not torch.equal(tr.model.means.detach()[:n0], mu0) or tr.model.n != n0, f'{app}: geometry must move after warm-up'
    if app == 'gc_log':
        A = torch.stack([v[0] for v in tr.gc.values()])
        assert abs(float(A.log().mean(0).abs().max())) < 1e-4, 'gauge'
    print(f'{app}: warm-up + {tr.model.n} Gaussians OK')
tmp = Path(tempfile.mkdtemp())
np.save(tmp / 'pts.npy', (np.random.default_rng(0).uniform(-0.3, 0.3, (500, 3))).astype(np.float32))
cfg = {'train': dict(iterations=20, densify_from=5, densify_until=15, densify_every=5, log_every=0, snapshots=[10]),
       'init': {'type': 'sfm', 'points': str(tmp / 'pts.npy'), 'n_points': 10, 'half_extent': 1.0}}
(tmp / 'seed_0').mkdir()
phase0.train_base(cfg, 0, pc, ras, bg, 'cpu', tmp / 'seed_0' / 'base.pt')
assert (tmp / 'seed_0' / 'snap_00010.pt').exists() and (tmp / 'seed_0' / 'base.pt').exists(), 'snapshot / base'
print('PATCH 5 OK (perm, noclip, oracle-clip, sat-mask, GC-ILA, warm-up, snapshot, SfM init, gsplat grad)')

## 3. Dữ liệu, tầng 0 và chuẩn bị

### 3.1 Scan DTU có sẵn
Bộ 15 scan chuẩn. Scan thiếu cần: `data/dtu/scanXX/{image, mask, cameras_sphere.npz}` (bản tiền xử lý định dạng
IDR/NeuS, cùng nguồn với scan24/63/83 hiện có) và `data/dtu/Points/stl/stlXXX_total.ply` (gói *Points* của DTU MVS 2014).
T2 tự chạy trên mọi scan có đủ dữ liệu; có thể thêm scan rồi chạy lại từ mục này.

In [ ]:
import yaml, json, time, glob, itertools, re, torch, numpy as np, pandas as pd
from pathlib import Path
DTU15 = (24, 37, 40, 55, 63, 65, 69, 83, 97, 105, 106, 110, 114, 118, 122)
def scan_ok(s):
    r = Path(f'data/dtu/scan{s}')
    return ((r / 'cameras_sphere.npz').exists() or (r / 'cameras.npz').exists()) and (r / 'image').is_dir() \
        and (r / 'mask').is_dir() and Path(f'data/dtu/Points/stl/stl{s:03d}_total.ply').exists()
AVAIL = [s for s in DTU15 if scan_ok(s)]
print('Có đủ dữ liệu:', AVAIL)
print('Thiếu:', [s for s in DTU15 if s not in AVAIL])

### 3.1b Tải dữ liệu DTU còn thiếu
- **Ảnh, mặt nạ, camera** (định dạng IDR: `image/`, `mask/`, `cameras.npz`; đúng 15 scan chuẩn): `DTU.zip` từ script
  `data/download_data.sh` của IDR (Yariv et al., 2020). Nếu link không tải được, dùng `dtu.tar.gz` trong thư mục dữ liệu
  của 2DGS (README của 2DGS, Google Drive): tải về Drive rồi điền đường dẫn vào `DTU_ARCHIVE_ON_DRIVE`.
- **Bề mặt tham chiếu STL**: `Points.zip` (khoảng 6.3 GB) từ trang chính thức DTU MVS 2014; chỉ giải nén file STL của
  các scan cần. Tải qua HTTP từ máy chủ DTU, có thể mất 30–60 phút.
Archive được tải vào ổ cục bộ `/content` (không chiếm Drive); chỉ thư mục scan và file STL được chép vào Drive.

In [ ]:
import shutil, zipfile, tarfile
DTU_URL = 'https://www.dropbox.com/s/ujmakiaiekdl6sh/DTU.zip?dl=1'           # IDR data/download_data.sh
DTU_ARCHIVE_ON_DRIVE = ''          # thay thế: ví dụ f'{STORE}/dtu.tar.gz' (bản của 2DGS)
POINTS_URL = 'http://roboimagedata2.compute.dtu.dk/data/MVS/Points.zip'   # DTU MVS 2014, "Points"

def has_scan(s):
    r = Path(f'data/dtu/scan{s}')
    return ((r / 'cameras_sphere.npz').exists() or (r / 'cameras.npz').exists()) and (r / 'image').is_dir() \
        and (r / 'mask').is_dir()
need_scan = [s for s in DTU15 if not has_scan(s)]
if need_scan:
    arc = DTU_ARCHIVE_ON_DRIVE or '/content/DTU.zip'
    if not DTU_ARCHIVE_ON_DRIVE and not os.path.exists(arc):
        subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', arc, DTU_URL], check=True)
    ex = '/content/dtu_src'
    if not os.path.isdir(ex):
        (zipfile.ZipFile(arc).extractall(ex) if arc.endswith('.zip') else tarfile.open(arc).extractall(ex))
    for s in need_scan:   # tên thư mục khác nhau giữa các bản phát hành: scan24, dtu_scan24, ...
        hits = sorted({os.path.dirname(p) for p in glob.glob(f'{ex}/**/cameras*.npz', recursive=True)
                       if re.search(rf'(^|_)scan{s}$', os.path.basename(os.path.dirname(p)))})
        if not hits:
            print(f'scan{s}: không có trong archive'); continue
        dst = Path(f'data/dtu/scan{s}')
        shutil.copytree(hits[0], dst, dirs_exist_ok=True)
        if not (dst / 'image').is_dir() and (dst / 'images').is_dir():
            (dst / 'images').rename(dst / 'image')
        n_img, n_msk = len(list((dst / 'image').glob('*.png'))), len(list((dst / 'mask').glob('*.png')))
        print(f'scan{s}: {n_img} ảnh, {n_msk} mặt nạ' + ('' if n_img == n_msk else '  <-- số lượng không khớp'))

need_stl = [s for s in DTU15 if not Path(f'data/dtu/Points/stl/stl{s:03d}_total.ply').exists()]
if need_stl:
    os.makedirs('data/dtu/Points/stl', exist_ok=True)
    if not os.path.exists('/content/Points.zip'):
        subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', '/content/Points.zip', POINTS_URL], check=True)
    with zipfile.ZipFile('/content/Points.zip') as z:
        names = z.namelist()
        for s in need_stl:
            name = next((n for n in names if n.endswith(f'stl{s:03d}_total.ply')), None)
            if name is None:
                print(f'stl{s:03d}: không có trong Points.zip'); continue
            with z.open(name) as f_in, open(f'data/dtu/Points/stl/stl{s:03d}_total.ply', 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f'stl{s:03d}_total.ply: xong')

AVAIL = [s for s in DTU15 if scan_ok(s)]
print('Có đủ dữ liệu:', AVAIL)
print('Thiếu:', [s for s in DTU15 if s not in AVAIL])

### 3.2 Tầng 0 – chỉ số xung đột I_conflict (không huấn luyện)

Với camera i, j: O_ij = tỉ lệ điểm bề mặt tham chiếu (lọc chặt) **nhìn thấy được** ở cả hai view (trong mặt nạ, qua kiểm tra
z-buffer). Với tham số nhiễu θ_k, D_ij = ‖log φ_i(Ī) − log φ_j(Ī)‖² tại cường độ trung bình Ī của vật thể.
**I_conflict = Σ O_ij D_ij / Σ O_ij**; đối chứng không trọng số I_unw = trung bình D_ij.
Cell này (i) tính I cho ba cấu hình nhiễu của E8 và đặt cạnh ΔW đã đo; (ii) duyệt toàn bộ 8! cách gán của cấu hình seed 0 và
chọn 4 hoán vị cho E-A (I nhỏ nhất, phân vị 1/3, 2/3, lớn nhất).

In [ ]:
from floater_metrics import load_eval_scene, project
from frecon.data.photometric import sample_params
dev = 'cuda'
STRONG = dict(exposure_stops=1.0, wb=0.15, gamma=0.20)
IU = np.triu_indices(8, 1)

def overlap_and_mean(s, N=8):
    cfg = yaml.safe_load(open(f'runs/photo_strong_dtu_scan{s}_v{N}/config.yaml', encoding='utf-8'))
    views, _, ref = load_eval_scene(cfg, dev, 'strict', 0.8, 16)
    vis = []
    for cam, m, zb in views:
        z, u, v, ok = project(ref, cam)
        vis.append((ok & m[v, u] & (z <= zb[v, u] + 0.02)).float())
    V = torch.stack(vis)
    O = (V @ V.T / V.shape[1]).cpu().numpy()
    Ibar = torch.stack([cam.image[m].mean(0) for cam, m, _ in views]).mean(0).cpu().numpy()
    return O, Ibar

def theta(params, Ibar):
    return np.array([p['gamma'] * (np.log(p['gain']) + np.log(np.array(p['wb'])) + np.log(Ibar)) for p in params])
def Dmat(th): return ((th[:, None] - th[None]) ** 2).sum(-1)
def I_conf(O, D): return float((O[IU] * D[IU]).sum() / O[IU].sum())

old = pd.read_csv(f'{RES_OLD}/final_with_delta.csv')
perms = np.array(list(itertools.permutations(range(8))))
PERMS, PERM_I, E8_I, rows = {}, {}, {}, []
for s in (63, 83):
    O, Ibar = overlap_and_mean(s)
    print(f'scan{s}: overlap O (off-diagonal) min {O[IU].min():.3f} max {O[IU].max():.3f}; Ī = {np.round(Ibar, 3)}')
    for ps, lvl in ((0, 'strong'), (1, 'strong_noise1'), (2, 'strong_noise2')):
        D = Dmat(theta(sample_params(8, seed=ps, **STRONG), Ibar))
        dw = old[(old.variant == 'base') & (old.N == 8) & (old.scan == f'scan{s}') & (old.level == lvl)].dW
        rows.append(dict(scan=f'scan{s}', noise_seed=ps, I_conflict=I_conf(O, D), I_unw=float(D[IU].mean()),
                         dW=dw.mean(), n=len(dw)))
        E8_I[f'scan{s}_seed{ps}'] = I_conf(O, D)
    D0 = Dmat(theta(sample_params(8, seed=0, **STRONG), Ibar))
    Dp = D0[perms[:, :, None], perms[:, None, :]]
    Ip = (O[IU] * Dp[:, IU[0], IU[1]]).sum(1) / O[IU].sum()
    order = np.argsort(Ip)
    pick = {'min': order[0], 'lo': order[len(order) // 3], 'hi': order[2 * len(order) // 3], 'max': order[-1]}
    PERMS[f'scan{s}'] = {k: perms[i].tolist() for k, i in pick.items()}
    PERM_I[f'scan{s}'] = {k: float(Ip[i]) for k, i in pick.items()} | {'identity': float(Ip[0])}
    print(f'  I theo hoán vị: min {Ip.min():.3f}, trung vị {np.median(Ip):.3f}, max {Ip.max():.3f}; '
          f'cách gán gốc (E8 seed 0) ở phân vị {(Ip < Ip[0]).mean():.2f}')
e8 = pd.DataFrame(rows)
print('\nE8: I_conflict và ΔW đã đo (mức strong, N = 8):\n', e8.round(4).to_string(index=False))
json.dump({'perms': PERMS, 'I': PERM_I, 'E8': E8_I}, open(f'{RES}/EA_perms.json', 'w'), indent=1)
print('\nHoán vị chọn cho E-A:', json.dumps(PERM_I, indent=1))

### 3.3 Khởi tạo SfM (triangulate từ ảnh huấn luyện **sạch**, pose đã biết)
SIFT trong mặt nạ → ghép cặp (ratio test) → kiểm tra epipolar với ma trận F từ pose → DLT → kiểm tra độ sâu dương, sai số
chiếu lại < 1.5 px, góc tam giác ≥ 2° → loại trùng theo voxel. Vị trí điểm giống nhau cho mọi mức nhiễu của cùng (scan, N),
nên so sánh theo cặp cô lập được tác động lên tối ưu 3DGS. Kiểm tra: khoảng cách trung vị tới bề mặt tham chiếu phải nhỏ
(cỡ 0.01 hoặc nhỏ hơn).

In [ ]:
from sfm_tri import scan_points
from frecon.gaussians import nearest_dist
for s in AVAIL:
    out = Path(f'data/dtu/scan{s}/sfm_v8.npy')
    if not out.exists():
        np.save(out, scan_points(f'data/dtu/scan{s}', 8))
    P = torch.from_numpy(np.load(out)).to(dev)
    cfg = yaml.safe_load(open('configs/dtu_scan24_v8.yaml', encoding='utf-8').read()
                         .replace('scan24', f'scan{s}').replace('stl024', f'stl{s:03d}'))
    load_eval_scene(cfg, 'cpu', 'default', 0.8, 16)   # tạo sẵn cache bề mặt tham chiếu mặc định (dùng khi huấn luyện)
    _, _, ref = load_eval_scene(cfg, dev, 'strict', 0.8, 16)
    d = nearest_dist(P, ref)
    print(f'scan{s}: {len(P)} điểm SfM; khoảng cách tới bề mặt: trung vị {d.median():.4f}, p90 {d.quantile(0.9):.4f}')

### 3.4 Kiểm tra backend gsplat (cho T2)
Cài gsplat (lần đầu biên dịch CUDA, có thể mất 5–10 phút), so sánh ảnh kết xuất gsplat với renderer PyTorch trên một checkpoint
cũ, và kiểm tra gradient vị trí 2D (dùng cho mật độ hoá) khác 0. Nếu không đạt, T2 tự chuyển sang renderer PyTorch (chậm hơn).
W_float luôn được đo bằng renderer PyTorch.

In [ ]:
from frecon.render import get_rasterizer
from frecon.train import TrainConfig, Trainer
GSPLAT_OK = False
try:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ninja', 'gsplat'], check=True)
    tras, gras, bg = get_rasterizer('torch'), get_rasterizer('gsplat'), torch.ones(3, device=dev)
    rd0 = 'runs/photo_none_dtu_scan63_v8'
    cfg = yaml.safe_load(open(f'{rd0}/config.yaml', encoding='utf-8'))
    views, test_cams, _ = load_eval_scene(cfg, dev, 'strict', 0.8, 16)
    model = Trainer.load(f'{rd0}/seed_0/base.pt', TrainConfig(**cfg['train']), tras, bg, dev).model
    ps = []
    with torch.no_grad():
        for c in test_cams[:4]:
            a, b = model.render(tras, c, bg)['image'], model.render(gras, c, bg)['image']
            ps.append(float(-10 * torch.log10(((a - b) ** 2).mean().clamp_min(1e-12))))
    out = model.render(gras, views[0][0], bg)
    out['means2d_full'].retain_grad(); out['image'].mean().backward()
    g = out['means2d_full'].grad
    GSPLAT_OK = min(ps) > 30 and g is not None and bool(torch.isfinite(g).all()) and float(g.abs().sum()) > 0
    print('PSNR giữa hai renderer:', np.round(ps, 1), '| gradient means2d:', None if g is None else float(g.abs().sum()))
except Exception as e:
    print('gsplat lỗi:', repr(e)[:300])
STD_RENDERER = 'gsplat' if GSPLAT_OK else 'torch'
print('GSPLAT_OK =', GSPLAT_OK, '-> T2 dùng renderer', STD_RENDERER)

## 4. Cấu hình thí nghiệm
Mức nhiễu như các đợt trước (mild, strong; seed nhiễu 0). Quy ước tên: `runs/v2_<mức><tags>_dtu_scan<s>_v<N>`, với tags:
`_hull` (visual hull), `_aff10` (affine học, lr 1e-2), `_gc` (GC-ILA), `_orc` (oracle), `_sat` (mặt nạ bão hoà),
`_noclip` (nhiễu không clip), `_warm1k` (đóng băng hình học và tắt mật độ hoá 1000 vòng đầu), `_p<min|lo|hi|max>`
(hoán vị E-A), `_std` (3DGS chuẩn: renderer gsplat, SH3, khởi tạo SfM, 10k vòng, mật độ hoá 500–5000).

In [ ]:
SEEDS = (0, 1, 2)
L3 = ('none', 'mild', 'strong')
S2 = (63, 83)
STD_RENDERER = globals().get('STD_RENDERER', 'torch')   # đặt ở mục 3.4
LEVELS = {'none': None, 'mild': dict(exposure_stops=0.5, wb=0.05, gamma=0.10, seed=0),
          'strong': dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=0)}
SNAPS = [250, 500, 750, 1000, 1500, 2000, 2500, 3000, 3500, 5000]
template = Path('configs/dtu_scan24_v8.yaml').read_text(encoding='utf-8')
PERMS = json.load(open(f'{RES}/EA_perms.json'))['perms'] if os.path.exists(f'{RES}/EA_perms.json') else {}
APP = {'affine10': '_aff10', 'gc_log': '_gc', 'oracle': '_orc'}

def make_cfg(s, N, level, init=None, app=None, sat=False, clip=True, warm=0, perm=None, std=False, snaps=False):
    tags = (f'_{init}' if init else '') + APP.get(app, '') + ('_sat' if sat else '') + ('' if clip else '_noclip') \
        + (f'_warm{warm // 1000}k' if warm else '') + (f'_p{perm}' if perm else '') + ('_std' if std else '')
    base = yaml.safe_load(template.replace('scan24', f'scan{s}').replace('stl024', f'stl{s:03d}'))
    cfg = dict(base, out_dir=f'runs/v2_{level}{tags}_dtu_scan{s}_v{N}', seeds=list(SEEDS), skip_signals=True)
    cfg['scene'] = dict(base['scene'], n_train_views=N)
    if LEVELS[level]:
        ph = dict(LEVELS[level])
        if not clip: ph['clip'] = False
        if perm: ph['perm'] = PERMS[f'scan{s}'][perm]
        cfg['scene']['photometric'] = ph
    tr = dict(base['train'])
    if app == 'affine10': tr.update(appearance='affine', lr_appearance=1e-2)
    elif app: tr['appearance'] = app
    if sat: tr['sat_mask'] = True
    if warm: tr['warmup_geom'] = warm
    if snaps: tr['snapshots'] = SNAPS
    if init == 'hull': cfg['init'] = dict(base['init'], type='hull')
    if std:
        cfg['renderer'] = STD_RENDERER
        cfg['init'] = dict(base['init'], type='sfm', points=f'data/dtu/scan{s}/sfm_v{N}.npy')
        tr.update(iterations=10000, densify_from=500, densify_until=5000, densify_every=100,
                  opacity_reset_every=3000, sh_degree=3, max_gaussians=300000)
    cfg['train'] = tr
    path = f"configs/{Path(cfg['out_dir']).name}.yaml"
    yaml.safe_dump(cfg, open(path, 'w', encoding='utf-8'), sort_keys=False)
    return path

def J(paths, seeds=SEEDS): return [(p, k) for p in paths for k in seeds]
SCHED = [(None, 0), (None, 1000), ('affine10', 0), ('affine10', 1000), ('gc_log', 0)]
EXPS = {
    'T2_std':     J([make_cfg(s, 8, l, std=True) for s in AVAIL for l in L3]),
    'E13_sat':    J([c for s in S2 for c in (make_cfg(s, 8, 'strong', app='oracle', clip=False),
                                             make_cfg(s, 8, 'strong', clip=False),
                                             make_cfg(s, 8, 'strong', app='oracle', sat=True),
                                             make_cfg(s, 8, 'none', sat=True))]),
    'M2_gcila':   J([make_cfg(s, 8, l, app='gc_log', sat=True) for s in S2 for l in L3]),
    'E12_timing': J([make_cfg(s, 8, l, init='hull', app=a, sat=(a == 'gc_log'), warm=w, snaps=True)
                     for s in S2 for a, w in SCHED for l in ('none', 'strong')]
                    + [make_cfg(s, 8, 'strong', init='hull', app='oracle', snaps=True) for s in S2]),
    'EA_perm':    J([make_cfg(s, 8, 'strong', perm=p) for s in S2 for p in ('min', 'lo', 'hi', 'max')]) if PERMS else [],
}
def out_dir(p): return yaml.safe_load(open(p, encoding='utf-8'))['out_dir']
def is_done(p, k): return os.path.exists(f'{out_dir(p)}/seed_{k}/base.pt')
for e, jobs in EXPS.items():
    print(f'{e:12s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)} đã có')

## 5. Huấn luyện song song (chạy lại được)
Thứ tự mặc định: T2 (nhanh, quyết định go/no-go) → E13 → M2 → E12 → E-A. Có thể sửa `RUN`. Log ở
`runs/<run>/seed_<k>/train.log`. Nếu runtime bị ngắt: chạy lại mục 1, 2, 3.1, 3.4, 4, 5 (mục 3.2, 3.3 đã lưu kết quả).
Ước lượng trên A100 với 7 tiến trình: khoảng 10 lượt/giờ cho run renderer PyTorch (E13, M2, E12, E-A: 132 lượt, khoảng
13 giờ); run T2 với gsplat nhanh hơn nhiều.

In [ ]:
RUN = ['T2_std', 'E13_sat', 'M2_gcila', 'E12_timing', 'EA_perm']
free_gb = torch.cuda.mem_get_info()[0] / 2**30
WORKERS = max(1, min(int(free_gb // 10), (os.cpu_count() or 8) - 1, 16))
print(f'GPU trống {free_gb:.0f} GB, CPU {os.cpu_count()} -> WORKERS = {WORKERS}')

def run_jobs(jobs, workers):
    env = dict(os.environ, PYTHONIOENCODING='utf-8', OMP_NUM_THREADS='2',
               PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    pending = [j for j in jobs if not is_done(*j)]
    print(f'{len(pending)} lượt cần chạy, {len(jobs) - len(pending)} đã có', flush=True)
    running, failed, t0 = [], [], time.time()
    while pending or running:
        while pending and len(running) < workers:
            p, k = pending.pop(0)
            d = f'{out_dir(p)}/seed_{k}'; os.makedirs(d, exist_ok=True)
            log = open(f'{d}/train.log', 'a')
            pr = subprocess.Popen([sys.executable, '-W', 'ignore', 'scripts/phase0.py', '--config', p,
                                   '--seeds', str(k), '--base_only'], stdout=log, stderr=subprocess.STDOUT, env=env)
            running.append((pr, p, k, log))
        time.sleep(15)
        for item in list(running):
            pr, p, k, log = item
            if pr.poll() is None:
                continue
            log.close(); running.remove(item)
            ok = pr.returncode == 0 and is_done(p, k)
            if not ok:
                failed.append((p, k))
            print(f'[{(time.time() - t0) / 60:6.1f} phút] {"OK  " if ok else "LỖI"} {Path(p).stem} seed {k} '
                  f'| chờ {len(pending)}, đang chạy {len(running)}', flush=True)
    print(f'xong sau {(time.time() - t0) / 60:.0f} phút; lỗi: {failed or "không"}')
    return failed

failed = run_jobs([j for e in RUN for j in EXPS[e]], WORKERS)

In [ ]:
# Tiến độ (hoặc từ terminal: tail -n 2 runs/v2_*/seed_*/train.log)
for e, jobs in EXPS.items():
    print(f'{e:12s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)}')
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 6. Đánh giá (renderer PyTorch, bề mặt tham chiếu lọc chặt, 16 view kiểm định chung)
Ba lượt: (a) mọi run v2 (W_float, W_float^Σ, PSNR trực tiếp và PSNR căn chỉnh); (b) snapshot của E12 (W_float theo vòng
lặp); (c) đánh giá lại checkpoint cũ với W_float^Σ và PSNR căn chỉnh (tầng 0; có thể bỏ qua, mục 7 sẽ dùng CSV cũ).

In [ ]:
def eval_parallel(dirs, tag, extra, workers):
    groups = {}
    for d in dirs:
        sc = yaml.safe_load(open(f'{d}/config.yaml', encoding='utf-8'))['scene']
        groups.setdefault((sc['root'], sc['n_train_views']), []).append(d)
    buckets = [[] for _ in range(min(workers, len(groups)))]
    for k in sorted(groups, key=lambda k: -len(groups[k])):
        min(buckets, key=len).extend(groups[k])
    procs = []
    for i, b in enumerate(buckets):
        out = f'{RES}/{tag}_part{i}.csv'; log = open(f'{RES}/{tag}_part{i}.log', 'w')
        procs.append((subprocess.Popen([sys.executable, '-W', 'ignore', 'scripts/fm2.py', '--run_dirs', *b,
                                        '--seeds', '0', '1', '2', '3', '4', '--out', out, *extra],
                                       stdout=log, stderr=subprocess.STDOUT), out, log))
    t0 = time.time()
    for pr, out, log in procs:
        pr.wait(); log.close()
        print(f'{out}: exit {pr.returncode} ({(time.time() - t0) / 60:.1f} phút)', flush=True)
    df = pd.concat([pd.read_csv(o) for _, o, _ in procs if os.path.exists(o)], ignore_index=True)
    df.to_csv(f'{RES}/{tag}.csv', index=False)
    return df

EVAL_WORKERS = max(1, min(WORKERS, 12))
has_ck = lambda d: bool(glob.glob(f'{d}/seed_*/base.pt'))
v2dirs = sorted(d for d in glob.glob('runs/v2_*_dtu_scan*_v*') if has_ck(d))
ev = eval_parallel(v2dirs, 'v2_final', [], EVAL_WORKERS)
snapdirs = [d for d in v2dirs if glob.glob(f'{d}/seed_*/snap_*.pt')]
evs = eval_parallel(snapdirs, 'v2_snaps', ['--snaps'], EVAL_WORKERS) if snapdirs else None
print(len(ev), 'dòng v2;', 0 if evs is None else len(evs), 'dòng snapshot')

In [ ]:
# (c) Tầng 0: đánh giá lại checkpoint cũ (336) với W_float^Σ và PSNR căn chỉnh
olddirs = sorted(d for d in glob.glob('runs/photo_*_dtu_scan*_v*') if has_ck(d))
oldv2 = eval_parallel(olddirs, 'old_v2eval', [], EVAL_WORKERS)
print(len(oldv2), 'dòng')

## 7. Phân tích và tiêu chí quyết định
`ΔW` = W_float(0.1) trừ mức none **cùng scan, seed, N và cùng cấu hình huấn luyện**. Thành phần chỉ thuộc về nhiễu
(`_orc`, `_noclip`, hoán vị, `_pix`, `_glob`) được bỏ khi tìm mức none; nếu run v2 không có mức none tương ứng thì dùng
run cũ cùng cấu hình (ví dụ oracle không clip so với base none cũ).

In [ ]:
from fm2 import parse
import scipy.stats as st
def load(p):
    d = pd.read_csv(p)
    if 'prefix' not in d:
        d = pd.concat([d, d.run.apply(parse).apply(pd.Series)], axis=1)
    return d
df = load(f'{RES}/v2_final.csv')
old = load(f'{RES}/old_v2eval.csv') if os.path.exists(f'{RES}/old_v2eval.csv') else load(f'{RES_OLD}/final_strict_pool16.csv')
df = pd.concat([df, old], ignore_index=True)
df['tags'] = df['tags'].fillna('')
for c in ('wsig_01', 'psnr_aligned'):
    if c not in df: df[c] = np.nan
NOISE_ONLY = re.compile(r'_orc|_noclip|_p(min|lo|hi|max)|_pix|_glob')
def add_delta(col):
    none = df[df.level == 'none'].groupby(['prefix', 'tags', 'scan', 'N', 'seed'])[col].mean()
    def f(r):
        t2 = NOISE_ONLY.sub('', r.tags)
        for p, t in ((r.prefix, r.tags), (r.prefix, t2), ('photo', t2)):
            k = (p, t, r.scan, r.N, r.seed)
            if k in none.index: return r[col] - none[k]
        return np.nan
    return df.apply(f, axis=1)
df['dW'], df['dWsig'] = add_delta('wfloat_01'), add_delta('wsig_01')
df['variant'] = df.prefix + ':' + df.tags.replace('', 'base')
df.to_csv(f'{RES}/v2_with_delta.csv', index=False)
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 400)
def summ(sub, by):
    return sub.groupby(by).agg(W=('wfloat_01', 'mean'), dW=('dW', 'mean'), sd=('dW', 'std'), n=('seed', 'count'),
                               up=('dW', lambda x: f'{(x > 0).sum()}/{x.notna().sum()}'), dWsig=('dWsig', 'mean'),
                               psnr=('psnr', 'mean'), psnr_al=('psnr_aligned', 'mean'), nG=('n_gaussians', 'mean')).round(3)
S2n = ['scan63', 'scan83']

In [ ]:
# T2 – go/no-go: 3DGS chuẩn (SH3, khởi tạo SfM, 10k vòng)
t2 = df[df.variant == 'v2:_std']
if len(t2):
    print(summ(t2, ['scan', 'level']).to_string())
    per = t2[t2.level == 'strong'].groupby('scan').dW.mean()
    frac, med = float((per > 0).mean()), float(per.median())
    print(f'\nΔW strong > 0 ở {frac:.0%} số scan ({len(per)} scan); trung vị {med:+.3f}')
    print('=> GO' if frac >= 0.8 and med >= 0.03 else '=> NO-GO / cần xem lại', '(tiêu chí: ≥ 80% scan dương và trung vị ≥ 0.03)')
    ref = df[(df.variant == 'photo:base') & (df.N == 8) & (df.level == 'strong')].groupby('scan').dW.mean()
    print('\nSo với pipeline cũ (SH0, khởi tạo ngẫu nhiên, mức strong):\n', pd.DataFrame({'chuẩn': per, 'cũ': ref}).dropna().round(3).to_string())

In [ ]:
# E13 – ngoại lệ oracle: clip hay không clip, mặt nạ bão hoà
v13 = ['photo:base', 'v2:_noclip', 'photo:_orc', 'v2:_orc_noclip', 'v2:_orc_sat']
e13 = df[df.variant.isin(v13) & (df.level == 'strong') & (df.N == 8) & df.scan.isin(S2n)]
print(summ(e13, ['scan', 'variant']).to_string())
print('\nΔW (strong):\n', e13.pivot_table(index='scan', columns='variant', values='dW').reindex(columns=v13).round(3).to_string())

In [ ]:
# M2 – GC-ILA so với các cách bù cũ (N = 8, khởi tạo ngẫu nhiên)
vm = ['photo:base', 'photo:_aff', 'photo:_aff10', 'photo:_cflog', 'photo:_orc', 'v2:_gc_sat']
m2 = df[df.variant.isin(vm) & (df.N == 8) & df.scan.isin(S2n)]
print(summ(m2, ['scan', 'level', 'variant']).to_string())
for col in ('dW', 'psnr', 'psnr_aligned'):
    sub = m2[m2.level != 'none'] if col == 'dW' else m2
    print(f'\n{col}:\n', sub
          .pivot_table(index=['scan', 'level'], columns='variant', values=col).reindex(columns=vm).round(3).to_string())
g = m2[(m2.level == 'strong')].groupby(['scan', 'variant'])[['dW', 'psnr']].mean().unstack('variant')
if ('dW', 'v2:_gc_sat') in g:
    r = (g[('dW', 'v2:_gc_sat')] / g[('dW', 'photo:base')]).round(2)
    dp = (g[('psnr', 'v2:_gc_sat')] - g[('psnr', 'photo:base')]).round(2)
    print('\nGC-ILA: tỉ lệ ΔW so với gốc', r.to_dict(), '| chênh PSNR trực tiếp (dB)', dp.to_dict(),
          '\n=> đạt' if (r <= 0.3).all() and (dp >= -1).all() else '\n=> chưa đạt', '(tiêu chí: tỉ lệ ≤ 0.3 và PSNR không thấp hơn quá 1 dB)')

In [ ]:
# E12 – thời điểm bù (visual hull): kết quả cuối
v12 = ['v2:_hull', 'v2:_hull_warm1k', 'v2:_hull_aff10', 'v2:_hull_aff10_warm1k', 'v2:_hull_gc_sat', 'v2:_hull_orc']
e12 = df[df.variant.isin(v12) & df.scan.isin(S2n)]
print(summ(e12, ['scan', 'variant', 'level']).to_string())
k = e12[e12.level == 'strong'].pivot_table(index='scan', columns='variant', values='dW').reindex(columns=v12)
print('\nΔW strong:\n', k.round(3).to_string())
if {'v2:_hull_aff10', 'v2:_hull_aff10_warm1k', 'v2:_hull', 'v2:_hull_warm1k'} <= set(k.columns):
    aff = 1 - k['v2:_hull_aff10_warm1k'] / k['v2:_hull_aff10']
    ctl = 1 - k['v2:_hull_warm1k'] / k['v2:_hull']
    print('\nMức giảm do warm-up: với affine', aff.round(2).to_dict(), '| đối chứng không affine', ctl.round(2).to_dict())
    print('=> ủng hộ cơ chế thời điểm' if (aff >= 0.5).all() and (aff - ctl >= 0.25).all() else '=> chưa ủng hộ',
          '(tiêu chí: giảm ≥ 50% với affine và hơn đối chứng ≥ 25 điểm phần trăm)')

In [ ]:
# E12 – quỹ đạo W_float(t) và ΔW(t) từ snapshot
import matplotlib.pyplot as plt
if os.path.exists(f'{RES}/v2_snaps.csv'):
    sn = load(f'{RES}/v2_snaps.csv'); sn['tags'] = sn['tags'].fillna('')
    sn['variant'] = sn.prefix + ':' + sn.tags.replace('', 'base')
    none_t = sn[sn.level == 'none'].set_index(['tags', 'scan', 'seed', 'step']).wfloat_01
    def dwt(r):
        for t in (r.tags, NOISE_ONLY.sub('', r.tags)):
            k_ = (t, r.scan, r.seed, r.step)
            if k_ in none_t.index: return r.wfloat_01 - none_t[k_]
        return np.nan
    s_ = sn[sn.level == 'strong'].copy(); s_['dW'] = s_.apply(dwt, axis=1)
    fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
    for j, sc in enumerate(S2n):
        for v in v12:
            a = s_[(s_.scan == sc) & (s_.variant == v) & (s_.step != 3000)].groupby('step')[['dW', 'wfloat_01']].mean()   # 3000: ngay sau reset độ mờ
            if len(a):
                axs[0, j].plot(a.index, a.dW, marker='o', ms=3, label=v.split(':')[1].lstrip('_'))
                axs[1, j].plot(a.index, a.wfloat_01, marker='o', ms=3, label=v.split(':')[1].lstrip('_'))
        for i_ in range(2):
            axs[i_, j].axvspan(300, 3500, color='0.9', zorder=0); axs[i_, j].grid(alpha=0.3)
        axs[0, j].set_title(f'{sc}: ΔW_float(t) (strong − none)'); axs[1, j].set_title(f'{sc}: W_float(t), strong')
        axs[1, j].set_xlabel('vòng lặp (vùng xám: mật độ hoá của lịch đồng thời)')
    axs[0, 0].legend(fontsize=8); fig.tight_layout(); fig.savefig(f'{RES}/E12_trajectories.png', dpi=150); plt.show()
    print(s_.pivot_table(index=['scan', 'step'], columns='variant', values='dW').round(3).to_string())

In [ ]:
# E-A – hoán vị cách gán nhiễu: I_conflict và ΔW
info = json.load(open(f'{RES}/EA_perms.json'))
rows = []
for sc in S2n:
    for p in ('min', 'lo', 'hi', 'max'):
        x = df[(df.variant == f'v2:_p{p}') & (df.scan == sc) & (df.level == 'strong')].dW
        rows.append(dict(scan=sc, config=f'perm_{p}', I=info['I'][sc][p], dW=x.mean(), n=len(x)))
    for ps, lvl in ((0, 'strong'), (1, 'strongs1'), (2, 'strongs2')):
        x = df[(df.variant == 'photo:base') & (df.scan == sc) & (df.level == lvl) & (df.N == 8)].dW
        rows.append(dict(scan=sc, config=f'E8_seed{ps}', I=info['E8'][f'{sc}_seed{ps}'], dW=x.mean(), n=len(x)))
ea = pd.DataFrame(rows).dropna(); print(ea.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 4))
for sc in S2n:
    a = ea[ea.scan == sc]
    if len(a) >= 3:
        rho = st.spearmanr(a.I, a.dW).statistic
        print(f'{sc}: Spearman ρ(I_conflict, ΔW) = {rho:+.2f} (n = {len(a)} cấu hình)')
        ax.scatter(a.I, a.dW, label=f'{sc} (ρ = {rho:+.2f})')
ax.set_xlabel('I_conflict'); ax.set_ylabel('ΔW_float (strong)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f'{RES}/EA_conflict.png', dpi=150); plt.show()

In [ ]:
# Tầng 0 – W_float^Σ có đổi kết luận? (biến thể gốc N = 8 và các cách bù)
b = df[(df.prefix == 'photo') & (df.N == 8) & df.dWsig.notna()]
if len(b):
    print(b[b.level != 'none'].pivot_table(index=['scan', 'level'], columns='variant', values=['dW', 'dWsig'])
          .round(3).to_string())
    print('\nTương quan W_float và W_float^Σ trên mọi checkpoint:', round(df[['wfloat_01', 'wsig_01']].corr().iloc[0, 1], 3))

In [ ]:
!ls -la {RES}